# Análisis Exploratorio de Datos (EDA)

El objetivo de este cuaderno es analizar la estructura, calidad y distribución de los datos extraídos en la capa cruda (raw). Se procederá a cargar los diferentes ficheros disponibles para comprobar si comparten un esquema tabular homogéneo, identificar variables útiles para la construcción del target predictivo y detectar posibles problemas de calidad (valores nulos o inconsistencias) antes de unificarlos.

In [1]:
import pandas as pd
import warnings

# Suprimir avisos de librerías para mantener la salida limpia
warnings.filterwarnings('ignore')

# Rutas de los ficheros de la capa raw
path_01 = "../data/raw/aa_dataset-tickets-multi-lang-5-2-50-version.csv"
path_02 = "../data/raw/dataset-tickets-multi-lang-4-20k.csv"
path_03 = "../data/raw/dataset-tickets-multi-lang3-4k.csv"

# Carga de datos
df_01 = pd.read_csv(path_01)
df_02 = pd.read_csv(path_02)
df_03 = pd.read_csv(path_03)

# Inspección de dimensiones
print(f"Dimensiones df_01: {df_01.shape}")
print(f"Dimensiones df_02: {df_02.shape}")
print(f"Dimensiones df_03: {df_03.shape}")

# Comparativa de esquemas (nombres de columnas)
print("\nEsquema df_01:", df_01.columns.tolist())
print("Esquema df_02:", df_02.columns.tolist())
print("Esquema df_03:", df_03.columns.tolist())

Dimensiones df_01: (28587, 16)
Dimensiones df_02: (20000, 15)
Dimensiones df_03: (4000, 17)

Esquema df_01: ['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']
Esquema df_02: ['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']
Esquema df_03: ['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'business_type', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8', 'tag_9']


Unificación de Datasets y Auditoría de Completitud

Tras identificar discrepancias en los esquemas originales, se procede a aislar la intersección de columnas relevantes para el modelado predictivo. Las variables de metadatos (etiquetas, versión, tipo de negocio) se descartan para evitar colapsos dimensionales y ruido. Una vez unificado el conjunto de datos, se ejecuta una auditoría de valores nulos y una exploración de la variable de idioma para fundamentar la viabilidad de las reglas de filtrado de negocio.

In [2]:
# Definición de la intersección de columnas estructurales requeridas
columnas_relevantes = ['subject', 'body', 'queue', 'type', 'priority', 'language']

# Proyección geométrica de los dataframes (Drop del ruido)
df_01_subset = df_01[columnas_relevantes]
df_02_subset = df_02[columnas_relevantes]
df_03_subset = df_03[columnas_relevantes]

# Unificación vertical
df_raw_unified = pd.concat([df_01_subset, df_02_subset, df_03_subset], ignore_index=True)

# Auditoría de completitud (Valores nulos)
nulos_totales = df_raw_unified.isnull().sum()
porcentaje_nulos = (nulos_totales / len(df_raw_unified)) * 100
df_nulos = pd.DataFrame({'Nulos Absolutos': nulos_totales, 'Porcentaje (%)': porcentaje_nulos})

print("Auditoría de Valores Nulos:")
print(df_nulos[df_nulos['Nulos Absolutos'] > 0])
print(f"\nTotal de registros unificados: {len(df_raw_unified)}")

# Distribución de la variable de idioma
print("\nDistribución de frecuencias por Idioma (%):")
print(df_raw_unified['language'].value_counts(normalize=True) * 100)

print("\nConteo absoluto por Idioma:")
print(df_raw_unified['language'].value_counts())

Auditoría de Valores Nulos:
         Nulos Absolutos  Porcentaje (%)
subject             5766       10.964687
body                   3        0.005705

Total de registros unificados: 52587

Distribución de frecuencias por Idioma (%):
language
en    56.386559
de    40.264704
es     1.544108
fr     0.905167
pt     0.899462
Name: proportion, dtype: float64

Conteo absoluto por Idioma:
language
en    29652
de    21174
es      812
fr      476
pt      473
Name: count, dtype: int64


# Saneamiento Estructural, Filtro Telco y Construcción de Capa Silver

En base a la auditoría de completitud, se aplica imputación sobre las ausencias en el asunto del ticket y eliminación estricta de registros con cuerpo corrupto o vacío. A continuación, se aísla el subconjunto de datos en inglés. 

**Pivotaje de Negocio (El Filtro Telco):** Para garantizar un entorno industrial acotado y evitar la dispersión de un BPO generalista hiper-ruidoso, aplicamos una regla de negocio estricta: amputamos las colas operativas basura (como Recursos Humanos o Soporte IT interno) y retenemos exclusivamente las 5 colas core de una compañía de telecomunicaciones. Esto reduce la entropía del sistema y define un dominio de negocio claro.

In [3]:
# Creación de la copia en memoria para la capa intermedia
df_silver = df_raw_unified.copy()

# Tratamiento de valores ausentes (Nulos)
df_silver['subject'] = df_silver['subject'].fillna('No Subject')
df_silver = df_silver.dropna(subset=['body'])

# Aplicación estricta de límite de negocio (Mercado Anglosajón)
df_silver = df_silver[df_silver['language'] == 'en']

# Limpieza de varianza cero
df_silver = df_silver.drop(columns=['language'])

# Aplicación estricta de filtro de negocio
colas_telco = ['Technical Support', 'Service Outages and Maintenance', 'Billing and Payments', 'Sales and Pre-Sales', 'Customer Service']
df_silver = df_silver[df_silver['queue'].isin(colas_telco)]

# Fusión de variables explicativas (Feature Engineering básico)
df_silver['full_text'] = df_silver['subject'] + ". " + df_silver['body']

# Corrección de Data Leakage: Deduplicación estricta por contenido
df_silver = df_silver.drop_duplicates(subset=['full_text'], keep='first')

# Auditoría de cardinalidad de las variables de tipificación
print("Dimensiones de la Capa Silver:", df_silver.shape)

print("\n--- Cardinalidad de las variables de tipificación ---")
print(f"Colas (queue): {df_silver['queue'].nunique()} valores únicos")
print(f"Tipos (type): {df_silver['type'].nunique()} valores únicos")
print(f"Prioridades (priority): {df_silver['priority'].nunique()} valores únicos")

print("\n--- Dominio de valores observados ---")
print("Valores en queue:", df_silver['queue'].unique().tolist())
print("Valores en type:", df_silver['type'].unique().tolist())
print("Valores en priority:", df_silver['priority'].unique().tolist())

Dimensiones de la Capa Silver: (15403, 6)

--- Cardinalidad de las variables de tipificación ---
Colas (queue): 5 valores únicos
Tipos (type): 4 valores únicos
Prioridades (priority): 3 valores únicos

--- Dominio de valores observados ---
Valores en queue: ['Technical Support', 'Billing and Payments', 'Sales and Pre-Sales', 'Service Outages and Maintenance', 'Customer Service']
Valores en type: ['Incident', 'Request', 'Problem', 'Change']
Valores en priority: ['high', 'low', 'medium']


In [4]:
import os

SILVER_DIR = "../data/processed"
os.makedirs(SILVER_DIR, exist_ok=True)

# Exportación estática de la Capa Silver
silver_path = os.path.join(SILVER_DIR, 'tickets_silver_cleaned.parquet')

# Usamos fastparquet para evitar el error de Arrow que detectamos
df_silver.to_parquet(silver_path, engine='fastparquet', index=False)

print(f"--- Capa Silver persistida con éxito ---")
print(f"Ruta: {silver_path} | Registros consolidados: {len(df_silver)}")

--- Capa Silver persistida con éxito ---
Ruta: ../data/processed\tickets_silver_cleaned.parquet | Registros consolidados: 15403


# Ingeniería de Target, Mapeo Anti-Falsificación y Exportación a Capa Gold

Se procede a la generación de la variable objetivo predictiva mediante la concatenación de la cola, tipo y prioridad (Tripleta). Al haber aplicado el filtro Telco en la capa anterior, la taxonomía se comprime naturalmente. 

Se ejecuta el mapeo Anti-Falsificación colapsando las clases minoritarias (soporte < 20) a la categoría estática `OUT_OF_SCOPE`. Esto previene que el particionado cruzado de Scikit-Learn colapse y asegura que el modelo no intente memorizar casuísticas marginales. El sistema escupe una topología final limpia de **56 clases**. Finalmente, se exporta el tensor resultante a formato Parquet para inyectarlo en los orquestadores de Machine Learning.

In [5]:
from sklearn.model_selection import train_test_split, StratifiedKFold
import os

# Configuración estricta MLOps V7.2
TARGET_COL = 'target_tripleta'
OUT_OF_SCOPE_LABEL = 'OUT_OF_SCOPE'
MIN_SAMPLES = 20
RANDOM_STATE = 42
GOLD_DIR = "../data/gold"

os.makedirs(GOLD_DIR, exist_ok=True)

df_gold = df_silver.copy()

# 1. Fusión del Target (La Tripleta)
df_gold[TARGET_COL] = (df_gold['queue'].astype(str) + '_' + 
                       df_gold['type'].astype(str) + '_' + 
                       df_gold['priority'].astype(str))

# 2. Auditoría de Soporte y Mapeo Anti-Falsificación Base
class_counts = df_gold[TARGET_COL].value_counts()
valid_classes = class_counts[class_counts >= MIN_SAMPLES].index

print("--- Auditoría Inicial de la Tripleta ---")
print(f"Total de combinaciones únicas generadas: {len(class_counts)}")
print(f"Clases retenidas (>= {MIN_SAMPLES} muestras): {len(valid_classes)}")

# Colapso de clases minoritarias automáticas
df_gold[TARGET_COL] = df_gold[TARGET_COL].apply(
    lambda x: x if x in valid_classes else OUT_OF_SCOPE_LABEL
)

out_of_scope_count = len(df_gold[df_gold[TARGET_COL] == OUT_OF_SCOPE_LABEL])
print(f"Tickets agrupados en '{OUT_OF_SCOPE_LABEL}': {out_of_scope_count} ({out_of_scope_count/len(df_gold)*100:.2f}%)")

# 3. Stratified Train/Test Split (80/20)
df_train, df_test = train_test_split(
    df_gold, 
    test_size=0.20, 
    random_state=RANDOM_STATE, 
    stratify=df_gold[TARGET_COL]
)

df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# 4. Pre-asignación Estática de Folds
df_train['fold_id'] = -1
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for fold, (train_idx, val_idx) in enumerate(skf.split(df_train, df_train[TARGET_COL])):
    df_train.loc[val_idx, 'fold_id'] = fold

# 5. Exportación a formato Parquet
train_path = os.path.join(GOLD_DIR, 'train_set_telco.parquet')
test_path = os.path.join(GOLD_DIR, 'test_set_telco.parquet')

df_train.to_parquet(train_path, index=False)
df_test.to_parquet(test_path, index=False)

print("\n--- Exportación a Capa Gold completada ---")
print(f"Train Set: {train_path} | Filas: {len(df_train)} | Clases Totales: {df_train[TARGET_COL].nunique()}")
print(f"Test Set:  {test_path} | Filas: {len(df_test)} | Clases Totales: {df_test[TARGET_COL].nunique()}")
print("Reproducibilidad garantizada.")

--- Auditoría Inicial de la Tripleta ---
Total de combinaciones únicas generadas: 60
Clases retenidas (>= 20 muestras): 55
Tickets agrupados en 'OUT_OF_SCOPE': 71 (0.46%)

--- Exportación a Capa Gold completada ---
Train Set: ../data/gold\train_set_telco.parquet | Filas: 12322 | Clases Totales: 56
Test Set:  ../data/gold\test_set_telco.parquet | Filas: 3081 | Clases Totales: 56
Reproducibilidad garantizada.
